* Contrast Stickiness vs. Listening Time
* Acquisition Rate vs. Retention

Findings
* Diff archetypes


Divide into... 
- Long term retention trends?
- Behavior (frequent but short sessions, infrequent but long sessions)
     >>> It makes sense that mobile might be shorter sessions, while usage where you're stuck in one place is going to be longer?


Carousel: for highlighting difference between MU and CU

- > Web... likely initial sign on platform, but not effective listening platform. Likely transferring to other platforms

-> Highlighht key difference between Android and diOS

-> DISTRIBUTION OF LISTENING IS INTERESTING

In [ ]:
import pandas as pd
import polars as pl
from plotnine import *

from polars import col, lit, when

import plotly.express as px

In [ ]:
def format_df_date_column(
        df: pl.DataFrame,
        date_col: str = "day",
) -> pl.DataFrame:
        
        df = (
                df.with_columns(
                        col(date_col).str.split("/").list.to_struct(fields=["Month", "Day", "Year"]).alias("unpacked")
                )
                .unnest("unpacked")
                .with_columns(
                        (lit("20") + col("Year")).alias("Year"),
                )
                .with_columns(
                        col("Month", "Day", "Year").cast(pl.Int64),
                        col(date_col).str.to_date("%m/%d/%y").alias("Date")
                )
                .with_columns(
                        pl.col("Date").dt.weekday().alias("DayOfWeek"),
                        pl.col("Date").dt.strftime("%A").alias("DayOfWeekStr"),
                )
                .with_columns(
                        when(col("DayOfWeek").is_in([6, 7]))
                        .then(lit("Weekend"))
                        .otherwise(lit("Weekday"))
                        .alias("WeekdayOrWeekend")
                )
                .rename({"day": "DayStr",})
        )
        return df


DF Setup

In [ ]:
df_mu = pl.read_csv("Data/Monthly_Listeners.txt", separator="\t")
df_du = pl.read_csv("Data/Daily_Listeners.txt", separator="\t")
df_ds = pl.read_csv("Data/Daily_Seconds_Listened.txt", separator="\t")
df_cu = pl.read_csv("Data/All-Time_Listeners.txt", separator="\t")

df_mu = (
    df_mu.with_columns(
        (
            pl.col("Year").cast(pl.Utf8) 
            + "-" 
            + pl.col("Month").cast(pl.Utf8).str.zfill(2)
        )
        .alias("YearMonth")
    )
    .with_columns(
        col("YearMonth").str.to_date("%Y-%m").alias("Date")
    )
    .rename({"Listeners": "NumListeners"})
)

df_du = (
    format_df_date_column(df_du)
    .rename({
        "platform": "Platform",
        "listeners": "NumListeners", 
    })
)

df_ds = (
    format_df_date_column(df_ds)
    .rename({
        "platform": "Platform",
        "seconds": "NumSecondsListened", 
    })
)


df_du_monthly_agg = (
    df_du.group_by("Platform", "Month", "Year")
    .agg(
        col("NumListeners").sum().alias("TotalDailyUniques"),
        col("NumListeners").mean().alias("MeanDailyUniques"),
    )
)

df_mu_du = (
    df_mu.join(
        df_du_monthly_agg,
        on=["Platform", "Month", "Year"],
        how="left"
    )
    .rename({"NumListeners": "MonthlyUniques"})
    .with_columns(
        (col("TotalDailyUniques")/col("MonthlyUniques")).alias("AvgDaysListened"),
        # Stickiness = On any given day, what percentage of this month's active users are using this product?
        #   e.g., 10% for January = On an average day in January, 10% of everyone who used the service at least once
        #     that month is active
        #   30 days X 10% = an average of 3 days per month
        (col("MeanDailyUniques")/col("MonthlyUniques")).alias("Stickiness"),
    )
)

df_du_ds = (
    df_du.join(
        df_ds, 
        on=["DayStr", "Platform", "Month", "Day", "Year", "Date", "DayOfWeek", "DayOfWeekStr", "WeekdayOrWeekend"], 
        how="left"
    )
    .with_columns((col("NumSecondsListened") / col("NumListeners")).alias("SecondsPerListener"))
)


df_ds_monthly_sum = (
    df_ds.group_by("Platform", "Month", "Year")
    .agg(col("NumSecondsListened").sum().alias("TotalSecondsListened"))
)
df_mu_du_ds = (
    df_mu_du.join(
        df_ds_monthly_sum, 
        on=["Platform", "Month", "Year"], 
        how="left"
    )
    .with_columns(
        (col("TotalSecondsListened") / col("TotalDailyUniques")).alias("AvgSecondsPerDAU"),
        (col("TotalSecondsListened") / col("MonthlyUniques")).alias("AvgSecondsPerMAU"),
    )

    # Create normalized version (share of total for each month) for Stickiness and Seconds listened metrics 
    # This will allow comparitive plotting between the different metrics
    .with_columns(
        (col("AvgDaysListened") / col("AvgDaysListened").sum().over(["Year", "Month"]))
            .alias("AvgDaysListened_Normalized"),
        (col("AvgSecondsPerMAU") / col("AvgSecondsPerMAU").sum().over(["Year", "Month"]))
            .alias("AvgSecondsPerMAU_Normalized")

    )
)


# Confirm that all normalized values sum to 1 for each month
#  (rounds to 7 to allow floating point tolerance)
assert all(x == 1 for x in df_mu_du_ds.group_by("Year", "Month").agg(col("AvgDaysListened_Normalized").sum().round(7))["AvgDaysListened_Normalized"].to_list())
assert all(x == 1 for x in df_mu_du_ds.group_by("Year", "Month").agg(col("AvgSecondsPerMAU_Normalized").sum().round(7))["AvgSecondsPerMAU_Normalized"].to_list())

df_mu_du_ds

df_cu = df_cu.rename({"Listeners": "CumulativeUniques"})

df_mu_cu = df_mu.join(
    df_cu,
    how="left",
    on=["Year", "Month", "Platform"]
)



In [ ]:
start_date = df_mu_cu["Date"].min()
end_date = df_mu_cu["Date"].max()

In [ ]:
pct_change_mus_cus = (
    df_mu_cu.filter(
        (col("Date") == col("Date").min().over("Platform")) |
        (col("Date") == col("Date").max().over("Platform"))
    )
    .select("YearMonth", "Platform", "NumListeners", "CumulativeUniques")
    .pivot(
        index="Platform",
        on="YearMonth"
    )
    .with_columns(
        ((col("NumListeners_2020-12") - col("NumListeners_2018-01"))/col("NumListeners_2018-01")).alias("PctChangeMAU"),
        ((col("CumulativeUniques_2020-12") - col("CumulativeUniques_2018-01"))/col("CumulativeUniques_2018-01")).alias("PctCumUniques"),
    )
)

pct_change_mus_cus


Focus on these statistics in case study?

In [ ]:
(
    pct_change_mus_cus
        .select("Platform", "PctCumUniques", "PctChangeMAU", )
        .filter(col("Platform").is_in(["iOS", "Android", "Web"]))
)

In [ ]:
(
    ggplot(data=df_mu_cu, mapping=aes(x="Date", y="NumListeners", color="Platform"))
    + geom_point(alpha=.5)
    + geom_smooth(method="lm", se=False)
    + theme_bw()
    + theme_bw()
)

In [ ]:
(
    ggplot(
        data=df_mu_du_ds,
        mapping=aes(x="Date", y="AvgDaysListened_Normalized", color="Platform")
    )
    + geom_line(alpha=.2)
    + geom_point(alpha=.2)
    + geom_smooth(linetype="dashed", se=False)

)

In [ ]:
df_mu

In [ ]:
df_mu_du_ds

In [ ]:
(
    ggplot(
        data=df_mu_du_ds,
        mapping=aes(x="Date", y="AvgSecondsPerMAU_Normalized", color="Platform")
    )
    + geom_line(alpha=.2)
    + geom_point(alpha=.2)
    + geom_smooth(linetype="dashed", se=False)

)

(
    ggplot(
        data=df_mu_du_ds,
        mapping=aes(x="Date", y="AvgSecondsPerMAU_Normalized", color="Platform")
    )
    + facet_wrap("Platform")
    + geom_smooth(linetype="dashed", se=False)
    + geom_smooth(mapping=aes(y="AvgDaysListened_Normalized"), linetype="dashed", se=False)

)

In [ ]:

# Averaging across each month in the dataset...
overall_avg_days_listened = df_mu_du_ds.group_by("Platform").agg(col("AvgDaysListened").mean()).sort("AvgDaysListened", descending=True)
overall_avg_seconds_per_dau = df_mu_du_ds.group_by("Platform").agg(col("AvgSecondsPerDAU").mean()).sort("AvgSecondsPerDAU", descending=True)

overall_avgs = overall_avg_days_listened.join(
    overall_avg_seconds_per_dau,
    how="left",
    on="Platform"
)

order_overall_avg_days_listened = (
    overall_avgs#.select(["Platform", "PctChangeWeekendVsWeekday"])
    # .unique(subset=["Platform"])
    .sort("AvgDaysListened", descending=True)
    .get_column("Platform")
    .to_list()
)


overall_avgs = overall_avgs.with_columns(
    col("Platform").cast(pl.Enum(categories=order_overall_avg_days_listened))
)

display(
    ggplot(data=overall_avgs, mapping=aes(x="Platform", y="AvgDaysListened"))
    + geom_col(),

    ggplot(data=overall_avgs, mapping=aes(x="Platform", y="AvgSecondsPerDAU"))
    + geom_col(),

)


In [ ]:
(
    ggplot(data=df_ds, mapping=aes(x="Date", y="NumSecondsListened", fill="Platform", color="Platform"))
    + geom_line()
)

(
    ggplot(data=df_du_ds, mapping=aes(x="Date", y="SecondsPerListener/60", fill="Platform", color="Platform"))
    + geom_line()
    # + geom_smooth(method="loess")
    # + scale_x_date(date_breaks="1 day", date_labels="%a")
    # + coord_cartesian(xlim=[pd.to_datetime("2018-01-01"), pd.to_datetime("2018-01-30")])
    + theme_bw()
    + theme(
       figure_size=[5, 4] 
    )
)

In [ ]:
(
    ggplot(data=df_mu_du_ds, mapping=aes(x="Date", y="AvgSecondsPerMAU/60", fill="Platform", color="Platform"))
    + facet_wrap("Platform")
    + geom_line()
    # + geom_smooth(method="loess")
    # + scale_x_date(date_breaks="1 day", date_labels="%a")
    # + coord_cartesian(xlim=[pd.to_datetime("2018-01-01"), pd.to_datetime("2018-01-30")])
    + theme_bw()
    + theme(
       figure_size=[5, 4] 
    )
)

In [ ]:
(
    ggplot(data=df_du, mapping=aes(x="Date", y="NumListeners", fill="Platform", color="Platform"))
    + geom_line()
    + scale_x_date(date_breaks="1 day", date_labels="%a")
    + coord_cartesian(xlim=[pd.to_datetime("2018-01-01"), pd.to_datetime("2018-01-30")])
    + theme_bw()
    + theme(
       figure_size=[14, 6] 
    )
)

In [ ]:
(
    df_du.group_by("DayOfWeekStr")
    .agg(col("NumListeners").mean())
)

df_du_by_day_of_week = (
    df_du.group_by("Platform", "DayOfWeek", "DayOfWeekStr")
    .agg(
        col("NumListeners").mean().alias("NumListenersMean"),
        col("NumListeners").std().alias("NumListenersStd"),
    )
)

df_du_by_weekend = (
    df_du.group_by("Platform", "WeekdayOrWeekend")
    .agg(
        col("NumListeners").mean().alias("NumListenersMean"),
        col("NumListeners").std().alias("NumListenersStd"),
    )
)

display(df_du_by_day_of_week, df_du_by_weekend)
(
    ggplot(data=df_du_by_day_of_week, mapping=aes(x="DayOfWeek", y="NumListenersMean"))
    + facet_wrap("Platform")
    + geom_col()
    + geom_errorbar(mapping=aes(ymin="NumListenersMean-NumListenersStd", ymax="NumListenersMean+NumListenersStd"))
)

(
    ggplot(data=df_du_by_weekend, mapping=aes(x="WeekdayOrWeekend", y="NumListenersMean"))
    + facet_wrap("Platform", scales="free_y")
    + geom_col()
    + geom_errorbar(mapping=aes(ymin="NumListenersMean-NumListenersStd", ymax="NumListenersMean+NumListenersStd"))
)

In [ ]:
df_du_by_weekend

In [ ]:
df_du_by_weekend_pct_change = (
    df_du_by_weekend.pivot(
        on="WeekdayOrWeekend",\
        index="Platform",
        values="NumListenersMean"
    )
    .select("Platform", "Weekday", "Weekend")
    .with_columns(
        (
            (col("Weekend") - col("Weekday")) 
            / col("Weekday")
        ).alias("PctChangeWeekendVsWeekday")
    )
    .with_columns(
        (col("PctChangeWeekendVsWeekday") > 0).alias("IsGainOnWeekend"),
        (
            (col("PctChangeWeekendVsWeekday") * 100)
                .round(1)
                .cast(pl.String) 
                + lit("%")
        ).alias("PctChangeLabel")
    )
)

order_platform_by_pct_change = (
    df_du_by_weekend_pct_change.select(["Platform", "PctChangeWeekendVsWeekday"])
    # .unique(subset=["Platform"])
    .sort("PctChangeWeekendVsWeekday", descending=True)
    .get_column("Platform")
    .to_list()
)


df_du_by_weekend_pct_change = df_du_by_weekend_pct_change.with_columns(
    col("Platform").cast(pl.Enum(categories=order_platform_by_pct_change))
)

df_du_by_weekend_pct_change

(
    ggplot(
        data=df_du_by_weekend_pct_change, 
        mapping=aes(x="Platform", y="PctChangeWeekendVsWeekday", fill="IsGainOnWeekend")
    )
    # + facet_wrap("Platform", scales="free_y")
    + geom_col()
    + geom_text(mapping=aes(label="PctChangeLabel"), nudge_y=.025, size=10)
    + scale_fill_manual({True: "green", False: "red"})
)

# platform_order_by_pct_change
# df_du_by_weekend_pct_change


In [ ]:
fig = px.bar(
    df_du_by_weekend_pct_change,
    x="Platform",
    y="PctChangeWeekendVsWeekday",
    color="IsGainOnWeekend",
    text="PctChangeLabel",
    color_discrete_map={True: "green", False: "red"},
)

fig.update_traces(textposition="outside")

fig.show()

In [ ]:

# Sort so bars stack/order correctly along the x-axis
df_mu = df_mu.sort(["Year", "Month"])
display(df_mu)
# Create stacked bar chart
fig = px.bar(
    df_mu,
    x="YearMonth",
    y="NumListeners",
    color="Platform",
    category_orders={"Platform": ["iOS", "Android",  "Web", "All_Other", "Roku", "Amazon",]}
)

fig.update_layout(
    barmode="stack", 
    bargap=0,
    xaxis_title="Year-Month", 
    yaxis_title="NumListeners",
    plot_bgcolor="white",
    paper_bgcolor="white"
)

fig.update_yaxes(
    showgrid=True,
    gridcolor="lightgray",
    gridwidth=1,
    griddash="dot",
    layer="above traces",
)

fig.show()

In [ ]:
fig.update_yaxes(
    range=[55_000_000, 70_000_000]
)

fig.show()

In [ ]:
fig.write_html(
    "test_plot.html",
    include_plotlyjs="cdn",
    full_html=True
)